# AI-Based Fall Detection System for Public Safety Surveillance

This Google Colab notebook implements a complete, self-contained, rule-based **Fall Detection System** using **YOLOv8-Pose** and custom IoU-based tracking. 

### Key Features:
1. **Pose Estimation**: Extracts 17 COCO keypoints and bounding boxes for humans using Ultralytics YOLOv8.
2. **IoU-Based Person Tracking**: Assigns and maintains tracking IDs across frames using a custom IoU matching algorithm (no external tracking libraries required).
3. **Rule-Based Fall Detection**: Employs geometric conditions (torso angle and aspect ratio) along with temporal persistence to identify falls and prevent false positives from crouching or bending.
4. **Alerting & Incident Logging**: Saves evidence frames for incidents and exports detailed CSV logs.
5. **Analytics Dashboard**: Dynamically visualizes incidents and compiles summary statistics.

---


## 1. Setup & Environment Configurations

In this section, we install the required packages, mount Google Drive to save logs and outputs permanently, create the necessary directory structures, and download the lightweight pre-trained YOLOv8 pose model (`yolov8n-pose.pt`).


In [ ]:
# Install required libraries
!pip install -q ultralytics opencv-python-headless pandas matplotlib

import os
import torch

# PyTorch 2.6 compatibility patch for loading YOLO models safely
try:
    _orig_load = torch.load
    def _patched_load(*args, **kwargs):
        kwargs['weights_only'] = False
        return _orig_load(*args, **kwargs)
    torch.load = _patched_load
except Exception:
    pass

from google.colab import drive
from ultralytics import YOLO

# 1. Mount Google Drive
try:
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully.")
    USE_DRIVE = True
except Exception as e:
    print(f"⚠️ Google Drive mount failed/skipped: {e}")
    print("Using local directory structure instead.")
    USE_DRIVE = False

# 2. Setup Folder Structure
if USE_DRIVE:
    BASE_DIR = "/content/drive/MyDrive/Task130_FallDetectionSystem"
else:
    BASE_DIR = "./Task130_FallDetectionSystem"

INPUT_DIR = os.path.join(BASE_DIR, "input_videos")
EVIDENCE_DIR = os.path.join(BASE_DIR, "outputs", "evidence_frames")
LOGS_DIR = os.path.join(BASE_DIR, "outputs", "incident_logs")

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(EVIDENCE_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

print("\nFolder structure initialized:")
print(f" - Input Videos: {INPUT_DIR}")
print(f" - Evidence Frames: {EVIDENCE_DIR}")
print(f" - Incident Logs: {LOGS_DIR}")
print("\nTeammates: Please place your input videos in the 'input_videos' folder above.")

# 3. Load YOLOv8 Pose Model
print("\nLoading YOLOv8-pose model...")
model = YOLO("yolov8n-pose.pt")
print("✅ YOLOv8 Pose model loaded successfully.")


## 2. Video & Image Sequence Input Handler

We define a robust video handler that reads input frames generator-style. To maximize compatibility, this handler supports **both standard video files (e.g. .mp4, .avi)** and **directories containing sequential images (e.g. .png, .jpg)**, which is common in research datasets like the UR Fall Detection Dataset. 

This approach is memory-efficient because it does not load the entire sequence into memory, and it uses safe OpenCV error handling to prevent crashes if files are corrupted or missing.


In [ ]:
import cv2
import os
import glob

def video_frame_generator(video_path):
    """
    Opens a video file OR a directory containing sequential images, and yields frames 
    sequentially along with sequence metadata.
    
    This function uses try/except blocks to ensure corrupted or invalid files 
    print a clear error message instead of crashing the notebook.
    
    Args:
        video_path (str): Path to the input video file or image sequence directory.
        
    Yields:
        tuple: (frame, frame_idx, fps, frame_count, width, height)
            - frame (numpy.ndarray): The current video frame.
            - frame_idx (int): The current frame index (0-based).
            - fps (float): Frame rate of the video.
            - frame_count (int): Total number of frames in the sequence.
            - width (int): Width of the video frame.
            - height (int): Height of the video frame.
            
    Raises:
        FileNotFoundError: If the video path or directory does not exist.
        IOError: If OpenCV fails to open the video stream or read the images.
    """
    if not os.path.exists(video_path):
        raise FileNotFoundError(f"Input path not found at: {video_path}")
        
    # Check if the path is a directory of images (e.g. frame-by-frame datasets)
    if os.path.isdir(video_path):
        image_extensions = ('*.png', '*.jpg', '*.jpeg', '*.bmp', '*.tif', '*.tiff')
        image_files = []
        for ext in image_extensions:
            # Match both lowercase and uppercase extensions
            image_files.extend(glob.glob(os.path.join(video_path, ext)))
            image_files.extend(glob.glob(os.path.join(video_path, ext.upper())))
            
        image_files = sorted(list(set(image_files)))
        
        if not image_files:
            raise FileNotFoundError(f"No image files found in directory: {video_path}")
            
        frame_count = len(image_files)
        # Read the first frame to determine video dimensions
        first_frame = cv2.imread(image_files[0])
        if first_frame is None:
            raise IOError(f"Could not read the first image frame: {image_files[0]}")
        height, width = first_frame.shape[:2]
        fps = 30.0 # Default fallback FPS for image sequences
        
        for frame_idx, img_path in enumerate(image_files):
            frame = cv2.imread(img_path)
            if frame is None:
                print(f"⚠️ Warning: Could not read frame image: {img_path}")
                continue
            yield frame, frame_idx, fps, frame_count, width, height
            
    else:
        # Standard video file input
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise IOError(f"OpenCV was unable to open the video file at: {video_path}")
            
        try:
            fps = cap.get(cv2.CAP_PROP_FPS)
            # Handle cases where FPS might be reported as 0 or invalid
            if fps <= 0 or fps is None:
                fps = 30.0
                
            frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            
            frame_idx = 0
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                yield frame, frame_idx, fps, frame_count, width, height
                frame_idx += 1
        finally:
            cap.release()


## 3. Pose Detection & Custom Tracking

This section runs YOLOv8-pose to extract human bounding boxes and their 17 COCO keypoints. To keep the notebook simple and avoid external compiled C-libraries, we implement a custom **Intersection-over-Union (IoU) Tracker** to assign consistent IDs to people across frames. 

We also define drawing functions to overlay bounding boxes, skeletons, and tracking IDs on the processed frames.


In [ ]:
import cv2
import numpy as np

def calculate_iou(box1, box2):
    """
    Computes the Intersection over Union (IoU) between two bounding boxes.
    
    Args:
        box1 (list or tuple): [x1, y1, x2, y2] bounding box coordinates.
        box2 (list or tuple): [x1, y1, x2, y2] bounding box coordinates.
        
    Returns:
        float: IoU value between 0.0 and 1.0.
    """
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    if x2 < x1 or y2 < y1:
        return 0.0
        
    intersection_area = (x2 - x1) * (y2 - y1)
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = box1_area + box2_area - intersection_area
    
    if union_area == 0.0:
        return 0.0
        
    return intersection_area / union_area

class SimpleIoUTracker:
    def __init__(self, iou_threshold=0.3, max_lost_frames=30):
        """
        A lightweight IoU-based tracker for matching human detections across frames.
        
        Args:
            iou_threshold (float): Minimum overlap required to match a detection to a track.
            max_lost_frames (int): Number of consecutive frames a track can be missing before deletion.
        """
        self.iou_threshold = iou_threshold
        self.max_lost_frames = max_lost_frames
        self.next_id = 1
        self.tracked_persons = {}
        
    def update(self, detections, frame_idx):
        """
        Updates the tracks with new detections from the current frame.
        
        Args:
            detections (list): List of dicts, each with keys "bbox", "confidence", "keypoints".
            frame_idx (int): The current frame index.
            
        Returns:
            dict: Currently active tracks visible in this frame.
        """
        active_ids = list(self.tracked_persons.keys())
        matches = []
        
        # Calculate IoU between all current detections and existing tracked persons
        for det_idx, det in enumerate(detections):
            det_bbox = det["bbox"]
            for track_id in active_ids:
                track_bbox = self.tracked_persons[track_id]["bbox"]
                iou = calculate_iou(det_bbox, track_bbox)
                if iou >= self.iou_threshold:
                    matches.append((iou, det_idx, track_id))
                    
        # Sort matches by IoU in descending order (greedy matching)
        matches.sort(key=lambda x: x[0], reverse=True)
        
        matched_det_indices = set()
        matched_track_ids = set()
        
        for iou, det_idx, track_id in matches:
            if det_idx in matched_det_indices or track_id in matched_track_ids:
                continue
                
            matched_det_indices.add(det_idx)
            matched_track_ids.add(track_id)
            
            # Update tracked person details
            det = detections[det_idx]
            track_data = self.tracked_persons[track_id]
            track_data["bbox"] = det["bbox"]
            track_data["keypoints"] = det["keypoints"]
            track_data["confidence"] = det["confidence"]
            track_data["lost_frames"] = 0
            track_data["last_seen_frame"] = frame_idx
            
            centroid_x = (det["bbox"][0] + det["bbox"][2]) / 2.0
            centroid_y = (det["bbox"][1] + det["bbox"][3]) / 2.0
            track_data["centroid_history"].append((centroid_x, centroid_y, frame_idx))
            
            if len(track_data["centroid_history"]) > 100:
                track_data["centroid_history"].pop(0)
                
        # Register new tracks for unmatched detections
        for det_idx, det in enumerate(detections):
            if det_idx not in matched_det_indices:
                centroid_x = (det["bbox"][0] + det["bbox"][2]) / 2.0
                centroid_y = (det["bbox"][1] + det["bbox"][3]) / 2.0
                
                self.tracked_persons[self.next_id] = {
                    "bbox": det["bbox"],
                    "keypoints": det["keypoints"],
                    "confidence": det["confidence"],
                    "centroid_history": [(centroid_x, centroid_y, frame_idx)],
                    "lost_frames": 0,
                    "is_fallen": False,
                    "last_fall_logged": False,
                    "fall_consecutive_frames": 0,
                    "last_seen_frame": frame_idx
                }
                self.next_id += 1
                
        # Handle lost tracks
        dead_tracks = []
        for track_id in active_ids:
            if track_id not in matched_track_ids:
                self.tracked_persons[track_id]["lost_frames"] += 1
                if self.tracked_persons[track_id]["lost_frames"] > self.max_lost_frames:
                    dead_tracks.append(track_id)
                    
        # Remove old lost tracks
        for track_id in dead_tracks:
            del self.tracked_persons[track_id]
            
        # Return tracks present in the current frame
        return {
            tid: data for tid, data in self.tracked_persons.items() 
            if data["last_seen_frame"] == frame_idx
        }

# COCO Keypoints Connections for Pose Visualizations
SKELETON_CONNECTIONS = [
    (0, 1), (0, 2), (1, 3), (2, 4),      # Head/Face
    (5, 6),                              # Shoulders midpoint link
    (5, 7), (7, 9),                      # Left arm
    (6, 8), (8, 10),                     # Right arm
    (5, 11), (6, 12),                    # Torso borders
    (11, 12),                            # Hips midpoint link
    (11, 13), (13, 15),                  # Left leg
    (12, 14), (14, 16)                   # Right leg
]

def draw_skeleton_and_box(frame, bbox, keypoints, person_id, is_fallen):
    """
    Overlays the bounding box, tracking ID, skeleton joints, and fall status on the frame.
    
    Args:
        frame (numpy.ndarray): OpenCV image frame.
        bbox (list): [x1, y1, x2, y2] coords.
        keypoints (numpy.ndarray): Pose keypoints of shape (17, 3) or (17, 2).
        person_id (int): Tracking ID assigned to the person.
        is_fallen (bool): Fall state indicator.
        
    Returns:
        numpy.ndarray: Annotated image frame.
    """
    x1, y1, x2, y2 = map(int, bbox)
    color = (0, 0, 255) if is_fallen else (0, 255, 0)
    thickness = 3 if is_fallen else 2
    
    # Draw box
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    
    # Label text
    status_str = "⚠️ FALLEN" if is_fallen else "NORMAL"
    label = f"ID {person_id} [{status_str}]"
    (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
    
    # Text background box
    cv2.rectangle(frame, (x1, y1 - 20), (x1 + w, y1), color, -1)
    cv2.putText(frame, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    
    num_cols = keypoints.shape[-1]
    
    # Draw connections
    for p1, p2 in SKELETON_CONNECTIONS:
        if p1 < len(keypoints) and p2 < len(keypoints):
            kp1, kp2 = keypoints[p1], keypoints[p2]
            conf1 = kp1[2] if num_cols == 3 else 1.0
            conf2 = kp2[2] if num_cols == 3 else 1.0
            
            if conf1 > 0.5 and conf2 > 0.5 and (kp1[0] != 0 or kp1[1] != 0) and (kp2[0] != 0 or kp2[1] != 0):
                pt1 = (int(kp1[0]), int(kp1[1]))
                pt2 = (int(kp2[0]), int(kp2[1]))
                cv2.line(frame, pt1, pt2, (255, 128, 0), 2)
                
    # Draw joint nodes
    for idx, kp in enumerate(keypoints):
        conf = kp[2] if num_cols == 3 else 1.0
        if conf > 0.5 and (kp[0] != 0 or kp[1] != 0):
            pt = (int(kp[0]), int(kp[1]))
            cv2.circle(frame, pt, 4, (0, 255, 255), -1)
            
    return frame


## 4. Rule-Based Fall Detection Logic

To distinguish a fall from common activities like bending down, crouching, or sitting, we use a robust **rule-based physical heuristic** rather than standard frame-based classifiers.

For each tracked person, we compute three metrics per frame:
1. **Torso Angle vs. Vertical Axis**: Evaluates the inclination of the torso vector (midpoint of shoulders to midpoint of hips). As the person falls, this angle shifts from near $0^\circ$ (vertical standing) to near $90^\circ$ (horizontal).
2. **Aspect Ratio (Width/Height)**: Compares bounding box proportions. Standing results in $AR < 0.5$ (narrower than tall). A horizontal fall flat leads to $AR \geq 1.0$ in ideal settings, but camera elevation and perspective angle compress the bounding box. Thus, we use a calibrated threshold of $0.6$ to accommodate various camera perspectives.
3. **Centroid Vertical Velocity (Change Rate)**: Measures the rate of downward centroid motion over a short window ($N$ frames), normalized by bounding box height to handle variable distance from the camera.

### The Fall Condition:
A fall is confirmed when:
$$\text{Torso Angle} \geq \text{Threshold}$$
$$\text{AND}$$
$$\text{Aspect Ratio} \geq \text{Threshold}$$
$$\text{AND}$$
$$\text{This state persists for at least } 1 \text{ second (persistence check)}$$

This persistence guard ensures that quick movements like bending down to pick up an item (where the aspect ratio remains small and the duration is brief) do not trigger false alerts.

All thresholds are exposed at the top of the cell for quick adjustments.


In [ ]:
import math

# ==========================================
# FALL DETECTION CONFIGURABLE THRESHOLDS
# ==========================================
FALL_ANGLE_THRESHOLD = 60.0       # Angle in degrees. Torso vs vertical (standing: ~0-30, falling: >60)
# Aspect ratio threshold: Bounding box width / height.
# In a perfect horizontal fall, aspect ratio >= 1.0. However, due to camera elevation
# and perspective compression, a value of 0.6 is used here to avoid missing falls.
FALL_ASPECT_RATIO_THRESHOLD = 0.6 
FALL_PERSISTENCE_SECONDS = 1.0    # Time in seconds the fall state must persist to confirm alert
VELOCITY_WINDOW_FRAMES = 5        # Number of frames (N) to compute vertical speed

def calculate_torso_angle(keypoints):
    """
    Computes the torso vector's angle in degrees relative to the vertical axis.
    Torso vector is drawn from the hips midpoint to the shoulders midpoint.
    
    Args:
        keypoints (numpy.ndarray): Pose joints of shape (17, 3) or (17, 2).
        
    Returns:
        float or None: Angle in degrees, or None if keypoints are missing/low confidence.
    """
    num_cols = keypoints.shape[-1]
    required_kps = [5, 6, 11, 12] # Left/Right shoulders, Left/Right hips
    
    if max(required_kps) >= len(keypoints):
        return None
        
    # Check confidences and invalid coords
    for idx in required_kps:
        conf = keypoints[idx][2] if num_cols == 3 else 1.0
        if conf <= 0.4 or (keypoints[idx][0] == 0 and keypoints[idx][1] == 0):
            return None
            
    # Shoulders midpoint
    s_x = (keypoints[5][0] + keypoints[6][0]) / 2.0
    s_y = (keypoints[5][1] + keypoints[6][1]) / 2.0
    
    # Hips midpoint
    h_x = (keypoints[11][0] + keypoints[12][0]) / 2.0
    h_y = (keypoints[11][1] + keypoints[12][1]) / 2.0
    
    # Torso vector components
    dx = s_x - h_x
    dy = s_y - h_y
    
    # Angle vs vertical axis (0 is vertical, 90 is horizontal)
    angle_rad = math.atan2(abs(dx), abs(dy))
    return math.degrees(angle_rad)

def calculate_aspect_ratio(bbox):
    """
    Computes the aspect ratio (width / height) of the bounding box.
    
    Args:
        bbox (list): [x1, y1, x2, y2] coords.
        
    Returns:
        float: Aspect ratio value.
    """
    x1, y1, x2, y2 = bbox
    width = max(0.0, x2 - x1)
    height = max(1e-6, y2 - y1)
    return width / height

def calculate_vertical_velocity(centroid_history, bbox_height, window_size=5):
    """
    Calculates vertical speed of the person, normalized by bbox height (scale-invariant).
    Positive values represent downward movement.
    
    Args:
        centroid_history (list): List of (x, y, frame_idx) tuples.
        bbox_height (float): Bounding box height in pixels.
        window_size (int): Temporal frame window size.
        
    Returns:
        float: Normalized vertical velocity.
    """
    if len(centroid_history) < 2:
        return 0.0
        
    curr_x, curr_y, curr_frame = centroid_history[-1]
    lookback_idx = max(0, len(centroid_history) - 1 - window_size)
    prev_x, prev_y, prev_frame = centroid_history[lookback_idx]
    
    frame_diff = curr_frame - prev_frame
    if frame_diff <= 0:
        return 0.0
        
    dy = curr_y - prev_y
    pix_velocity = dy / frame_diff
    
    # Normalize by bounding box height
    return pix_velocity / max(1e-6, bbox_height)


## 5. Alerting & Incident Logging

This section manages alerting and incident documentation. To prevent spamming the log files with duplicate alerts while a person is already down, we implement a **state-transition guard**. An incident is logged **only once** on the transition from a non-fallen state to a fallen state.

### Actions on Fall:
1. **Evidence Capture**: The exact video frame is saved as a JPEG to `/outputs/evidence_frames/` using the format `incident_{timestamp}_{person_id}.jpg`.
2. **Alert Console**: Prints a warning alert in the console: `⚠️ FALL DETECTED - Person {id} at frame {n}`.
3. **Structured Log**: Appends a row containing metadata (timestamp, tracking ID, YOLO pose confidence, frame number, evidence image filename) to a pandas DataFrame.
4. **CSV Export**: At the end of video processing, the DataFrame is written to `incident_log.csv` under `/outputs/incident_logs/`.


In [ ]:
import datetime
import os
import cv2
import pandas as pd

# Global dataframe logs format:
# Columns: ['timestamp', 'person_id', 'confidence_score', 'frame_number', 'evidence_filename', 'video_name']

def log_incident(frame, frame_idx, person_id, confidence, video_name, evidence_dir, log_list):
    """
    Triggers an alert, saves the evidence frame, and logs the incident details.
    
    Args:
        frame (numpy.ndarray): Frame image where the fall was detected.
        frame_idx (int): The video frame index.
        person_id (int): Tracking ID of the person.
        confidence (float): YOLO detection confidence score.
        video_name (str): Filename of the processed video.
        evidence_dir (str): Folder path to save the JPEG snapshot.
        log_list (list): Reference to the list of log dicts to append to.
        
    Returns:
        str: Filename of the saved evidence snapshot.
    """
    # 1. Print visual console warning
    print(f"⚠️ FALL DETECTED - Person {person_id} at frame {frame_idx}")
    
    # 2. Format timestamp and filename
    timestamp_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    evidence_filename = f"incident_{timestamp_str}_{person_id}.jpg"
    evidence_path = os.path.join(evidence_dir, evidence_filename)
    
    # 3. Save the snapshot frame (OpenCV BGR write)
    cv2.imwrite(evidence_path, frame)
    
    # 4. Append to list
    log_list.append({
        "timestamp": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "person_id": person_id,
        "confidence_score": float(confidence),
        "frame_number": frame_idx,
        "evidence_filename": evidence_filename,
        "video_name": video_name
    })
    
    return evidence_filename


## 6. Analytics Dashboard

This cell builds a graphical **Analytics Dashboard** using `matplotlib` and `pandas`. It operates independently by reading the generated CSV log and a small JSON statistics file from the logs directory.

### Visualizations & Metrics:
1. **Incident Count Chart**: Bar plot showing the number of falls detected across sessions/videos.
2. **Timeline Scatter Plot**: Maps fall events along the video duration timeline (frame number vs. Person ID). Points are sized and colored according to YOLO detection confidence.
3. **Key Stats Table**: Prints total incidents, total unique people tracked, and average detection confidence.


In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt

def display_analytics_dashboard(logs_dir):
    """
    Loads incident records from the CSV file and summary statistics, 
    and displays a formatted analytical dashboard.
    
    Args:
        logs_dir (str): Folder containing incident logs and statistics.
    """
    csv_path = os.path.join(logs_dir, "incident_log.csv")
    summary_path = os.path.join(logs_dir, "summary_stats.json")
    
    if not os.path.exists(csv_path):
        print("❌ No incident log CSV found. Process a video first.")
        return
        
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"❌ Error loading incident log CSV: {e}")
        return
        
    print("\n" + "="*60)
    print("             FALL DETECTION MONITORING ANALYTICS")
    print("="*60)
    
    # Display printed stats
    if os.path.exists(summary_path):
        try:
            with open(summary_path, 'r') as f:
                stats = json.load(f)
            print(f"Total Incident Alerts Triggered: {stats.get('total_incidents', 0)}")
            print(f"Total Unique People Tracked:      {stats.get('total_people_tracked', 0)}")
            print(f"Average Detection Confidence:    {stats.get('avg_confidence', 0.0):.2f}")
        except Exception as e:
            print(f"Warning: Could not read summary_stats.json: {e}")
    else:
        # Fallback if summary stats JSON is absent
        print(f"Total Incident Alerts Triggered: {len(df)}")
        if len(df) > 0:
            print(f"Total Unique People Fallen:      {df['person_id'].nunique()}")
            print(f"Average Detection Confidence:    {df['confidence_score'].mean():.2f}")
        else:
            print("Total Unique People Fallen:      0")
            print("Average Detection Confidence:    N/A")
            
    print("="*60 + "\n")
    
    if len(df) == 0:
        print("Zero incidents logged. Skip plotting.")
        return
        
    # Render Plots
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Plot 1: Falls per Video/Session
    if 'video_name' in df.columns:
        video_counts = df['video_name'].value_counts()
        axes[0].bar(video_counts.index, video_counts.values, color='#4A90E2', edgecolor='black', zorder=2)
        axes[0].set_title("Incident Alert Count per Video Session", fontsize=12, fontweight='bold')
        axes[0].set_xlabel("Video File Name", fontsize=10)
        axes[0].set_ylabel("Number of Incidents", fontsize=10)
        axes[0].tick_params(axis='x', rotation=45)
        axes[0].grid(axis='y', linestyle='--', alpha=0.5, zorder=1)
    else:
        axes[0].bar(["Session 1"], [len(df)], color='#4A90E2', edgecolor='black')
        axes[0].set_title("Incident Counts", fontsize=12, fontweight='bold')
        axes[0].grid(axis='y', linestyle='--', alpha=0.5)
        
    # Plot 2: Timeline Scatter Plot
    # X: frame_number, Y: person_id, Color/Size: confidence_score
    scatter = axes[1].scatter(
        df['frame_number'], 
        df['person_id'].astype(str), 
        s=df['confidence_score'] * 350, 
        c=df['confidence_score'], 
        cmap='plasma', 
        edgecolors='black', 
        alpha=0.85,
        zorder=2
    )
    axes[1].set_title("Incident Timeline (Frame vs. Person ID)", fontsize=12, fontweight='bold')
    axes[1].set_xlabel("Frame Number", fontsize=10)
    axes[1].set_ylabel("Tracked Person ID", fontsize=10)
    axes[1].grid(True, linestyle='--', alpha=0.5, zorder=1)
    
    cbar = fig.colorbar(scatter, ax=axes[1])
    cbar.set_label('YOLO Pose Confidence Score', fontsize=9)
    
    plt.tight_layout()
    plt.show()

# Run the dashboard visually if the CSV exists
display_analytics_dashboard(LOGS_DIR)


## 7. End-to-End Runner

This final cell connects all parts of the system. You specify the input video path `VIDEO_PATH` at the top of the cell. 

### Processing Pipeline:
1. Opens the input video sequence using the input frame generator.
2. Initializes the `cv2.VideoWriter` to record the output video.
3. Loops through frames, running pose detection and updating the IoU tracking states.
4. Evaluates the fall posture rules and triggers alerts on state transitions.
5. Overlays pose skeleton lines, labels, and bounding boxes on the frames.
6. Records progress, exports the final CSV log + JSON stats, prints performance metrics (Total Time, Average FPS), and automatically calls the dashboard.


In [ ]:
import os
import time
import json
import pandas as pd
import cv2

# =====================================================================
# CONFIGURATION CELL - EDIT THESE TO RUN ON YOUR VIDEOS
# =====================================================================
# Teammates: upload your video to Task130_FallDetectionSystem/input_videos/
# and edit the path below:
VIDEO_PATH = os.path.join(INPUT_DIR, "test_surveillance.mp4")

# Path to write the annotated output video
OUTPUT_VIDEO_PATH = os.path.join(BASE_DIR, "outputs", "annotated_output.mp4")
# =====================================================================

def run_fall_detection_pipeline(video_path, output_video_path):
    """
    Executes the end-to-end fall detection, tracking, logging, and video saving.
    
    Args:
        video_path (str): Path to input video file.
        output_video_path (str): Path to save the output video.
    """
    print(f"🎬 Initializing pipeline for: {video_path}")
    
    # 1. Verification of video existence
    if not os.path.exists(video_path):
        print(f"❌ Error: Video file not found at: {video_path}")
        print("Please place your video file in the input directory or update the path above.")
        print(f"Expected Input Directory: {INPUT_DIR}")
        return
        
    start_time = time.time()
    processed_frames = 0
    total_tracked_people = set()
    incident_logs = []
    
    # Create output directory if not present
    os.makedirs(os.path.dirname(output_video_path), exist_ok=True)
    # Strip any trailing slashes to correctly extract folder name if path is a directory
    video_filename = os.path.basename(video_path.rstrip('/\')) or "image_sequence"
    
    # Reset tracker for a clean session
    tracker = SimpleIoUTracker(iou_threshold=0.3, max_lost_frames=30)
    
    # 2. Main processing loop
    try:
        frame_generator = video_frame_generator(video_path)
        out_writer = None
        
        for frame, frame_idx, fps, frame_count, width, height in frame_generator:
            if out_writer is None:
                # Setup output video writer
                fourcc = cv2.VideoWriter_fourcc(*'mp4v')
                out_writer = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))
                print(f"Video specs: {width}x{height} pixels | {fps:.2f} FPS | {frame_count} total frames")
                print("Processing frames...")
                
            # Run YOLO pose detection
            results = model.predict(frame, verbose=False)
            result = results[0]
            
            detections = []
            if result.boxes is not None and len(result.boxes) > 0:
                boxes = result.boxes.xyxy.cpu().numpy()
                confs = result.boxes.conf.cpu().numpy()
                classes = result.boxes.cls.cpu().numpy()
                
                kps = result.keypoints.data.cpu().numpy() if result.keypoints is not None else None
                
                for idx in range(len(boxes)):
                    # COCO class 0 is Person
                    if int(classes[idx]) == 0:
                        detections.append({
                            "bbox": list(boxes[idx]),
                            "confidence": float(confs[idx]),
                            "keypoints": kps[idx] if kps is not None else None
                        })
                        
            # Update IoU tracker
            current_tracks = tracker.update(detections, frame_idx)
            
            # Record unique tracked IDs
            for tid in current_tracks.keys():
                total_tracked_people.add(tid)
                
            # Loop tracks to run detection logic
            annotated_frame = frame.copy()
            for track_id, track in current_tracks.items():
                if track["keypoints"] is None:
                    continue
                    
                # Compute features
                torso_angle = calculate_torso_angle(track["keypoints"])
                bbox = track["bbox"]
                aspect_ratio = calculate_aspect_ratio(bbox)
                
                # Check fall criteria
                is_instantly_falling = (
                    torso_angle is not None 
                    and torso_angle >= FALL_ANGLE_THRESHOLD 
                    and aspect_ratio >= FALL_ASPECT_RATIO_THRESHOLD
                )
                
                if is_instantly_falling:
                    track["fall_consecutive_frames"] += 1
                    # Persistence frame count
                    required_frames = max(1, int(fps * FALL_PERSISTENCE_SECONDS))
                    
                    if track["fall_consecutive_frames"] >= required_frames:
                        was_fallen = track["is_fallen"]
                        track["is_fallen"] = True
                        
                        # Logging transition state
                        if not was_fallen and not track["last_fall_logged"]:
                            log_incident(
                                frame=frame,
                                frame_idx=frame_idx,
                                person_id=track_id,
                                confidence=track["confidence"],
                                video_name=video_filename,
                                evidence_dir=EVIDENCE_DIR,
                                log_list=incident_logs
                            )
                            track["last_fall_logged"] = True
                else:
                    # Reset track states if they stand up
                    track["fall_consecutive_frames"] = 0
                    track["is_fallen"] = False
                    track["last_fall_logged"] = False
                    
                # Annotate frame
                annotated_frame = draw_skeleton_and_box(
                    annotated_frame, 
                    track["bbox"], 
                    track["keypoints"], 
                    track_id, 
                    track["is_fallen"]
                )
                
            # Write annotated frame
            out_writer.write(annotated_frame)
            processed_frames += 1
            
            # Print feedback
            if processed_frames % 50 == 0 or processed_frames == frame_count:
                progress = (processed_frames / frame_count) * 100
                print(f" ⏳ Processed {processed_frames}/{frame_count} frames ({progress:.1f}%)")
                
        # Clean up
        if out_writer is not None:
            out_writer.release()
            
        if processed_frames == 0:
            print("❌ Processing finished, but no frames were loaded. Check video file.")
            return
            
        # 3. Export CSV Log
        if len(incident_logs) == 0:
            log_df = pd.DataFrame(columns=["timestamp", "person_id", "confidence_score", "frame_number", "evidence_filename", "video_name"])
        else:
            log_df = pd.DataFrame(incident_logs)
            
        csv_path = os.path.join(LOGS_DIR, "incident_log.csv")
        
        # If running multiple times, we can preserve history by appending or reading
        if os.path.exists(csv_path) and os.path.getsize(csv_path) > 0:
            try:
                existing_df = pd.read_csv(csv_path)
                log_df = pd.concat([existing_df, log_df], ignore_index=True)
                log_df = log_df.drop_duplicates(subset=["timestamp", "person_id", "frame_number", "video_name"])
            except Exception:
                pass
                
        log_df.to_csv(csv_path, index=False)
        print(f"📊 Incident log CSV written/updated: {csv_path}")
        
        # 4. Save statistics summary JSON
        avg_conf = log_df['confidence_score'].mean() if len(log_df) > 0 else 0.0
        summary_stats = {
            "total_incidents": len(log_df),
            "avg_confidence": float(avg_conf),
            "total_people_tracked": len(total_tracked_people)
        }
        summary_path = os.path.join(LOGS_DIR, "summary_stats.json")
        with open(summary_path, 'w') as f:
            json.dump(summary_stats, f, indent=4)
            
        # 5. Timing Stats
        elapsed = time.time() - start_time
        avg_fps = processed_frames / elapsed
        
        print("\n" + "="*60)
        print("🎉 PIPELINE RUN COMPLETED SUCCESSFULLY")
        print("="*60)
        print(f"Total Processing Time: {elapsed:.2f} seconds")
        print(f"Average FPS Achieved:  {avg_fps:.2f} FPS")
        print(f"Annotated Video Saved: {output_video_path}")
        print("="*60 + "\n")
        
        # Draw Dashboard
        display_analytics_dashboard(LOGS_DIR)
        
    except Exception as e:
        print(f"❌ Critical pipeline failure: {e}")
        import traceback
        traceback.print_exc()

# Run the pipeline
run_fall_detection_pipeline(VIDEO_PATH, OUTPUT_VIDEO_PATH)


## Summary & Technical Documentation

### Model Information
- **Model Used**: Ultralytics YOLOv8-pose Nano (`yolov8n-pose.pt`)
- **Keypoints Format**: COCO Pose representation containing 17 keypoint nodes (joints and facial landmarks).
- **Inference Hardware**: Runs optimally on Google Colab's standard T4 GPU (accelerated inference) but will run on standard CPU instances.

### Approach Summary
This system implements a **rule-based physical heuristic** rather than a machine-learning-based temporal classifier (like an LSTM or GRU). The heuristic monitors three main variables:
1. Torso tilt angle relative to the vertical axis (using shoulder and hip coordinates).
2. Aspect ratio of the bounding box.
3. Centroid vertical velocity.

A fall event is triggered if the torso angle is flat, the aspect ratio is horizontal, and this state persists for 1.0 second. This approach is simple, fast, highly interpretable, and does not require a large amount of annotated temporal training data.

### Known Limitations
- **Occlusions**: If a person is partially blocked by furniture, walls, or other people, keypoints for the shoulders or hips might not be visible or will be inaccurately estimated, which can prevent the torso angle check from working.
- **Overlapping People**: The simple IoU tracker relies on spatial overlap. If two people walk close to each other or cross paths, their IDs may be swapped.
- **Low Light / Poor Contrast**: Vision-based pose estimators degrade in performance under poor lighting conditions, leading to missing keypoints.
- **Camera Angles**: Highly elevated cameras (e.g., top-down birds-eye view) compress the aspect ratio and vertical orientation, rendering horizontal comparisons less effective.

### Suggested Version 2 Improvements
1. **Temporal ML Classifier**: Integrate an **LSTM** (Long Short-Term Memory) or **GRU** neural network that takes the sequence of 17 keypoint coordinates over time (e.g., 30 frames) to classify actions, handling complex falls (e.g. slow collapses, stumbling) more reliably.
2. **Robust Multi-Object Tracker**: Swap the simple IoU tracker with **ByteTrack** or **OC-SORT** (both supported out-of-the-box by Ultralytics) to maintain tracking IDs across long occlusions and crowded scenes.
3. **3D Pose Estimation**: Utilize a 3D pose estimator (like MediaPipe or Lift3D) to extract depth information, making fall heuristics independent of the camera angle.
4. **Adaptive Thresholds**: Automatically adjust aspect ratio and torso thresholds based on the camera perspective angle and person distance.
